# CSE427 Milestone 2 — Finalized EDA and Preprocessing Pipeline

**Course:** CSE427
**Project Title:** Evidence-Aware Multi-Agent Retrieval-Augmented Generation for Scientific Literature Review
**Milestone:** Milestone 2
**Purpose:** This notebook finalizes the exploratory data analysis (EDA) and preprocessing pipeline for the QASPER scientific literature dataset. It validates dataset characteristics, completes structured data preparation, and generates processed representations required for downstream retrieval and RAG experiments.

## 1. Environment Setup

We install the necessary libraries for data processing, visualization, and dataset handling.


In [1]:
import os
import sys
from pathlib import Path

# Detect Google Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/raiyanhossainaraf-dagger/CSE427Labproject_S3_G10.git"
    PROJECT_NAME = "CSE427LabProject_S3_G10"
    PROJECT_ROOT = Path("/content") / PROJECT_NAME
    
    # Use branch/ref if provided in environment
    GIT_REF = os.environ.get("CSE427_GIT_REF")
    
    if not PROJECT_ROOT.exists():
        if GIT_REF:
            !git clone --branch {GIT_REF} --single-branch {REPO_URL} {PROJECT_ROOT}
        else:
            !git clone {REPO_URL} {PROJECT_ROOT}
    
    os.chdir(PROJECT_ROOT)
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
else:
    # Local execution
    # Try to find root from current working directory or upward
    current_dir = Path.cwd().resolve()
    # Check if we are in notebooks/
    if (current_dir / "src").exists():
        PROJECT_ROOT = current_dir
    elif (current_dir.parent / "src").exists():
        PROJECT_ROOT = current_dir.parent
    else:
        # Fallback to searching upward
        temp_root = current_dir
        while temp_root.parent != temp_root:
            if (temp_root / "src").exists():
                break
            temp_root = temp_root.parent
        PROJECT_ROOT = temp_root

    os.chdir(PROJECT_ROOT)
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))

# Import central configuration
from src.config import get_project_paths, ensure_project_directories

PATHS = get_project_paths(explicit_root=PROJECT_ROOT)
ensure_project_directories(PATHS)

# Backward compatibility
PROJECT_ROOT = PATHS.project_root
DATA_DIR = PATHS.data_dir
RAW_DATA_DIR = PATHS.raw_data_dir
PROCESSED_DATA_DIR = PATHS.processed_data_dir
OUTPUT_DIR = PATHS.outputs_dir
FIGURE_DIR = PATHS.figures_dir
TABLE_DIR = PATHS.tables_dir
SUMMARY_DIR = PATHS.summaries_dir

print(f"Environment initialized. Project root: {PROJECT_ROOT}")

Running locally
Project root: C:\ARAF\Python\CSE427\CSE427LabProject_S3_G10
Current directory: C:\ARAF\Python\CSE427\CSE427LabProject_S3_G10


In [2]:
print("Project contents:")
for item in PROJECT_ROOT.iterdir():
    print(" -", item.name)

print("\nsrc contents:")
for item in (PROJECT_ROOT / "src").iterdir():
    print(" -", item.name)

Project contents:
 - .git
 - .gitignore
 - .idea
 - .venv
 - data
 - notebooks
 - outputs
 - README.md
 - report
 - requirements.txt
 - src

src contents:
 - bm25_retrieval.py
 - data_loader.py
 - eda_utils.py
 - embeddings.py
 - evaluation.py
 - hybrid_retrieval.py
 - preprocessing.py
 - rag_baseline.py
 - retrieval.py
 - utils.py
 - vector_store.py
 - __init__.py
 - __pycache__


In [3]:
from src.data_loader import (
    load_qasper_dataset,
    download_qasper_fallback
)

from src.preprocessing import *
from src.eda_utils import *
from src.utils import *

print("src modules imported successfully!")

src modules imported successfully!


## 2. Configuration

We define the directory structure and global constants.


In [5]:
# Global Constants
RANDOM_SEED = 42

from src.utils import set_seed
set_seed(RANDOM_SEED)

print("Configuration complete.")

## 3. Load QASPER Dataset

We load the QASPER dataset from the official JSON files. A fallback downloader and extractor are included for stability.


In [6]:
import importlib
import src.data_loader


importlib.reload(src.data_loader)

from src.data_loader import (
    load_qasper_dataset,
    download_qasper_fallback,
)

print("Checking for QASPER dataset archives...")

download_qasper_fallback(RAW_DATA_DIR)

print("\nLoading QASPER dataset from official local JSON files...")
ds = load_qasper_dataset(RAW_DATA_DIR)

if not ds or not any(ds.values()):
    raise RuntimeError(
        "QASPER dataset could not be loaded. "
        "Check data/raw/qasper_v0.3 and the source archives."
    )

print("\nDataset loaded successfully.")
for split, split_data in ds.items():
    print(f"Split '{split}': {len(split_data):,} papers")

# Split integrity checks
train_ids = set(ds["train"].keys())
val_ids = set(ds["validation"].keys())
test_ids = set(ds["test"].keys())

print(f"\nIntegrity Checks:")
print(f"Train/Validation overlap: {len(train_ids & val_ids)}")
print(f"Train/Test overlap: {len(train_ids & test_ids)}")
print(f"Validation/Test overlap: {len(val_ids & test_ids)}")

assert len(train_ids & val_ids) == 0, "Overlap found between Train and Validation!"
assert len(train_ids & test_ids) == 0, "Overlap found between Train and Test!"
assert len(val_ids & test_ids) == 0, "Overlap found between Validation and Test!"

Checking for QASPER dataset archives...
Extracting train_dev.tgz...
Successfully extracted train_dev.tgz
Extracting test.tgz...
Successfully extracted test.tgz

Loading QASPER dataset from official local JSON files...
Loaded 888 papers for split: train
Loaded 281 papers for split: validation
Loaded 416 papers for split: test

Dataset loaded successfully.
Split 'train': 888 papers
Split 'validation': 281 papers
Split 'test': 416 papers

Integrity Checks:
Train/Validation overlap: 0
Train/Test overlap: 0
Validation/Test overlap: 0


### Inspect Dataset Structure

We examine a sample record to understand the nested schema.


In [7]:
if ds and 'train' in ds and ds['train']:
    split_name = 'train'
    # ds[split_name] is a dict keyed by paper ID
    sample_paper_id = list(ds[split_name].keys())[0]
    sample_paper = ds[split_name][sample_paper_id]
    # Ensure ID is in the object for consistent processing
    sample_paper['id'] = sample_paper_id
    
    print(f"Available splits: {list(ds.keys())}")
    print(f"Fields in a paper: {list(sample_paper.keys())}")
    print(f"Example Title: {sample_paper['title']}")
    # Nested structure inspection
    print(f"Number of sections: {len(sample_paper['full_text'])}")

Available splits: ['train', 'validation', 'test']
Fields in a paper: ['title', 'abstract', 'full_text', 'qas', 'figures_and_tables', 'id']
Example Title: Minimally Supervised Learning of Affective Events Using Discourse Relations
Number of sections: 21


## 4. Dataset Description

**Dataset Name:** QASPER (Question Answering on Scientific Papers)  
**Source:** Allen Institute for AI  
**Domain:** Scientific Research (NLP)  
**Format:** Nested JSON/Dataset object  

**Attributes:**
- `id`: Unique paper identifier.
- `title`: Paper title.
- `abstract`: Paper abstract.
- `full_text`: List of sections, each containing paragraphs.
- `qas`: Questions associated with the paper, containing multiple answer annotations and supporting evidence paragraphs.

**Relevance:** QASPER is ideal because it provides human-annotated evidence for every answer. This allows us to train and evaluate the evidence-aware retrieval component of our RAG system.


## 5. Create Structured DataFrames

We parse the nested QASPER structure into three primary DataFrames: Paper-level, Question-level, and Evidence-level. This simplifies downstream analysis and ensures that all annotations are preserved.


In [8]:
from src.preprocessing import create_structured_dataframes

print("Converting QASPER JSON to structured DataFrames...")
dfs = create_structured_dataframes(ds)

papers_df = dfs["papers"]
questions_df = dfs["questions"]
evidence_df = dfs["evidence"]

print(f"Papers: {len(papers_df):,}")
print(f"Questions: {len(questions_df):,}")
print(f"Evidence Items: {len(evidence_df):,}")

# Save processed tables
papers_df.to_parquet(PROCESSED_DATA_DIR / "papers.parquet", index=False)
questions_df.to_parquet(PROCESSED_DATA_DIR / "questions.parquet", index=False)
evidence_df.to_parquet(PROCESSED_DATA_DIR / "evidence.parquet", index=False)

print("\nDataFrames saved to data/processed/")

# Validation
assert not papers_df.empty, "Papers DataFrame is empty!"
assert not questions_df.empty, "Questions DataFrame is empty!"
assert not evidence_df.empty, "Evidence DataFrame is empty!"

display(papers_df.head(2))

Converting QASPER JSON to structured DataFrames...
Papers: 1,585
Questions: 5,049
Evidence Items: 12,761

DataFrames saved to data/processed/


,paper_id,split,title,abstract,num_sections,num_paragraphs,full_text_word_count,num_questions
0,1909.00694,train,Minimally Supervised Learning of Affective Eve...,Recognizing affective events that trigger posi...,21,72,2475,9
1,2003.07723,train,"PO-EMO: Conceptualization, Annotation, and Mod...",Most approaches to emotion analysis regarding ...,25,80,5667,3


## 6. Preprocessing and Chunking

We implement section-aware flattening and chunking. The RAG system will search these chunks rather than complete papers. We use a transparent chunking method that preserves section boundaries.


In [9]:
from src.preprocessing import process_papers_to_chunks

CHUNK_SIZE_WORDS = 350
CHUNK_OVERLAP_WORDS = 50

print(f"Chunking papers (Size: {CHUNK_SIZE_WORDS}, Overlap: {CHUNK_OVERLAP_WORDS})...")
chunks_df = process_papers_to_chunks(ds, CHUNK_SIZE_WORDS, CHUNK_OVERLAP_WORDS)

print(f"Total chunks generated: {len(chunks_df):,}")

# Save chunks
chunks_df.to_parquet(PROCESSED_DATA_DIR / "chunks.parquet", index=False)
print("Chunks saved to data/processed/chunks.parquet")

# Validation
assert chunks_df["chunk_id"].is_unique, "Duplicate chunk IDs found!"
assert chunks_df["text"].str.strip().ne("").all(), "Empty chunks found!"
assert chunks_df["paper_id"].notna().all(), "Chunks missing paper_id!"

display(chunks_df.head(2))

Chunking papers (Size: 350, Overlap: 50)...
Total chunks generated: 29,360
Chunks saved to data/processed/chunks.parquet


,chunk_id,paper_id,paper_title,split,section_name,chunk_index,text,word_count
0,1909.00694_0_0,1909.00694,Minimally Supervised Learning of Affective Eve...,train,Introduction,0,Affective events BIBREF0 are events that typic...,350
1,1909.00694_0_1,1909.00694,Minimally Supervised Learning of Affective Eve...,train,Introduction,1,to be of the same polarity (for Cause) or of t...,83


## 7. Generate Embeddings

We use the `sentence-transformers/all-MiniLM-L6-v2` model to generate dense vector representations for each text chunk. These embeddings will be used for similarity search in the retrieval phase.


In [10]:
from src.embeddings import load_embedding_model, generate_embeddings, save_embeddings

# Load model (automatically detects GPU if available)
model = load_embedding_model()

# Generate embeddings for all chunks with optimized batch size
chunk_texts = chunks_df["text"].tolist()
embeddings = generate_embeddings(chunk_texts, model, batch_size=64)

# Save embeddings and metadata mapping
save_embeddings(embeddings, PROCESSED_DATA_DIR / "chunk_embeddings.npy")

# Save metadata mapping (index -> chunk_id)
chunk_metadata = chunks_df[["chunk_id", "paper_id", "paper_title", "section_name"]].copy()
chunk_metadata.to_parquet(PROCESSED_DATA_DIR / "chunk_metadata.parquet", index=False)

print(f"Embeddings shape: {embeddings.shape}")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 on device: cuda...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generating embeddings for 29360 items (Batch Size: 64)...


Batches:   0%|          | 0/459 [00:00<?, ?it/s]

Embeddings saved to C:\ARAF\Python\CSE427\chunk_embeddings.npy
Embeddings shape: (29360, 384)


## 8. Build FAISS Vector Database

We construct a FAISS index to enable efficient similarity search over the generated chunk embeddings.


In [11]:
from src.vector_store import create_faiss_index, save_faiss_index

# Create FAISS index
index = create_faiss_index(embeddings)

# Save index to disk
FAISS_PATH = PROCESSED_DATA_DIR / "faiss_index"
FAISS_PATH.mkdir(parents=True, exist_ok=True)
save_faiss_index(index, FAISS_PATH / "index.faiss")

FAISS index created with 29360 vectors.
FAISS index saved to C:\ARAF\Python\CSE427\faiss_index\index.faiss


## 9. Retrieval Testing

We test the retrieval system by querying it with questions from the QASPER dataset and inspecting the top-k retrieved chunks.


In [ ]:
from src.retrieval import retrieve_relevant_chunks, display_retrieval_results

# Example question from the dataset or custom
sample_q = questions_df.iloc[0]["question"]
print(f"Testing retrieval for question: {sample_q}")

# Retrieve
retrieved_chunks = retrieve_relevant_chunks(sample_q, model, index, chunks_df, top_k=5)

# Display results
display_retrieval_results(sample_q, retrieved_chunks)

Testing retrieval for question: What is the seed lexicon?
Generating embeddings for 1 items (Batch Size: 64)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

## 10. Baseline RAG Preparation

We prepare the prompt structure for the future RAG system by formatting the retrieved context along with the question.


In [13]:
from src.rag_baseline import prepare_baseline_rag

# Prepare RAG prompt
rag_prompt = prepare_baseline_rag(sample_q, retrieved_chunks)

--- BASELINE RAG PROMPT PREPARED ---

Use the following retrieved scientific context to answer the question. 
If the answer is not in the context, state that you do not have enough information.
Always cite the source numbers in your answer.

Context:
Source 1 (Paper: Minimally Supervised Learning of Affective Events Using Discourse Relations, Section: Proposed Method ::: Discourse Relation-Based Event Pairs ::: AL (Automatically Labeled Pairs)):
The seed lexicon matches (1) the latter event but (2) not the former event, and (3) their discourse relation type is Cause or Concession. If the discourse relation type is Cause, the former event is given the same score as the latter. Likewise, if the discourse relation type is Concession, the former event is given the opposite of the latter's score. They are used as reference scores during training.

Source 2 (Paper: Minimally Supervised Learning of Affective Events Using Discourse Relations, Section: Proposed Method ::: Discourse Relation-Bas

## Findings
- **Data Scaling:** Processing 1,585 papers resulted in approximately 30,000 retrievable chunks.
- **Retrieval Performance:** The FAISS index with L2 distance provides fast and relevant context retrieval for scientific queries.
- **Evidence Mapping:** Human-annotated evidence is preserved in `evidence.parquet` for future verification.
